# v2 Data Audit and Cohort — Step 2

This notebook builds `df_model_v2` from the already-validated `df_model_v1.parquet`
baseline. It performs no modeling — it audits schema, derives the survey-design and
risk-stratification columns approved in the v2 protocol revision, runs the
`s434`/`s435` missingness audit, reproduces all validated counts, and saves the
frozen v2 dataset and its variable dictionary.

Nothing below hardcodes result numbers — every count, percentage, and table is
computed live from the loaded parquet file.

## 1. Imports and paths

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

pd.set_option("display.max_columns", None)

V1_PATH = Path("../../data/processed/df_model_v1.parquet")
V2_OUT_PATH = Path("../../data/processed/df_model_v2.parquet")
DICT_OUT_PATH = Path("../../data/processed/variable_dictionary_v2.csv")

## 2. Load v1 and verify schema

In [2]:
df = pd.read_parquet(V1_PATH)

n_v1 = len(df)
resp_v1 = df["respondent_id"].nunique()

print("Loaded shape:", df.shape)
print("Unique respondents:", resp_v1)

Loaded shape: (201311, 49)
Unique respondents: 158429


In [3]:
# Every column this notebook depends on must be present before anything else runs.
required_columns = [
    "respondent_id", "birth_index", "birth_order", "sample_weight",
    "cluster_number", "primary_sampling_unit",
    "sample_stratum_v022", "sample_stratum_v023",
    "state", "district", "facility_type", "csection",
    "maternal_age", "twin_order",
    "convulsions_during_pregnancy", "swelling_during_pregnancy",
    "education_years", "wealth_index", "residence", "social_group",
    "anc_visits",
]

missing_required = [c for c in required_columns if c not in df.columns]
assert not missing_required, f"Missing required columns: {missing_required}"
print("All required columns present.")

All required columns present.


## 3. Survey-design verification

Confirms — rather than assumes — whether `cluster_number`/`primary_sampling_unit`
are identical, and whether `sample_stratum_v022`/`sample_stratum_v023` are
identical or genuinely distinct. Neither is hardcoded; both are computed here.

In [4]:
psu_identical = (df["cluster_number"] == df["primary_sampling_unit"]).all()
print("cluster_number == primary_sampling_unit for all rows:", psu_identical)
if not psu_identical:
    n_diff = (df["cluster_number"] != df["primary_sampling_unit"]).sum()
    print(f"  Mismatched rows: {n_diff} — do not assume interchangeability.")

strata_identical = (df["sample_stratum_v022"] == df["sample_stratum_v023"]).all()
print("\nsample_stratum_v022 == sample_stratum_v023 for all rows:", strata_identical)
print("v022 unique values:", df["sample_stratum_v022"].nunique(),
      "| range:", df["sample_stratum_v022"].min(), "-", df["sample_stratum_v022"].max())
print("v023 unique values:", df["sample_stratum_v023"].nunique(),
      "| range:", df["sample_stratum_v023"].min(), "-", df["sample_stratum_v023"].max())
print("\nNote: even if identical numerically, confirm against the NFHS-5 recode "
      "manual which variable is the officially documented sampling-error stratum "
      "before citing it in the methods section.")

cluster_number == primary_sampling_unit for all rows: True

sample_stratum_v022 == sample_stratum_v023 for all rows: True
v022 unique values: 2709 | range: 11 - 93223
v023 unique values: 2709 | range: 11 - 93223

Note: even if identical numerically, confirm against the NFHS-5 recode manual which variable is the officially documented sampling-error stratum before citing it in the methods section.


## 4. `education_years` special-code check

In [5]:
n_inconsistent = (df["education_years"] == 97).sum()
print(f"education_years == 97 ('inconsistent') records: {n_inconsistent}")

df["education_years"] = df["education_years"].replace(97, np.nan)
print("Cleaning applied (no-op if the count above was 0).")

education_years == 97 ('inconsistent') records: 0
Cleaning applied (no-op if the count above was 0).


## 5. Derived columns: survey weight and first-birth flag

`first_birth` uses `birth_order` (`bord`), **not** `birth_index` (`bidx`). See the
documentation section at the end of this notebook for why.

In [6]:
df["sample_weight_normalized"] = df["sample_weight"] / 1_000_000
df["first_birth"] = df["birth_order"] == 1

n_first = df["first_birth"].sum()
print(f"First births: {n_first} ({n_first / len(df) * 100:.2f}% of cohort)")

First births: 82607 (41.03% of cohort)


## 6. Primary risk strata — fully observed variables only

This is the **headline** stratification used for the primary heterogeneity
analysis. It uses only `maternal_age` and `twin_order`, both of which have zero
missingness in this cohort, so no records are excluded for missing data.

Terminology: **"lower-risk"**, not "low-risk" — NFHS-5's available variables do
not capture comprehensive obstetric risk, so the label is deliberately hedged.

In [7]:
def primary_risk_stratum(row):
    """Fully-observed-variable stratum for first births.

    Returns 'not_applicable_multiparous' for any birth_order > 1.
    Never depends on prior-C-section history (not derivable — see documentation).
    """
    if not row["first_birth"]:
        return "not_applicable_multiparous"
    is_singleton = row["twin_order"] == 0
    is_young = row["maternal_age"] < 35
    return "lower_risk" if (is_young and is_singleton) else "elevated_risk"


df["risk_stratum_primary"] = df.apply(primary_risk_stratum, axis=1)

primary_counts = df["risk_stratum_primary"].value_counts()
primary_pct = df["risk_stratum_primary"].value_counts(normalize=True) * 100

print("Primary stratum counts and percentages:")
for stratum in primary_counts.index:
    print(f"  {stratum}: {primary_counts[stratum]} ({primary_pct[stratum]:.2f}%)")

n_unclassified_primary = (
    df["first_birth"] & ~df["risk_stratum_primary"].isin(["lower_risk", "elevated_risk"])
).sum()
print(f"\nUnclassified first births in primary scheme: {n_unclassified_primary} "
      f"(expected: 0, since only fully-observed variables are used)")

Primary stratum counts and percentages:
  not_applicable_multiparous: 118704 (58.97%)
  lower_risk: 79701 (39.59%)
  elevated_risk: 2906 (1.44%)

Unclassified first births in primary scheme: 0 (expected: 0, since only fully-observed variables are used)


## 7. Refined complication-based strata — SENSITIVITY ANALYSIS ONLY

Incorporates `s434` (convulsions) and `s435` (swelling) where both are observed.
**This stratification is never the primary result** — see the missingness audit
in the next section for why.

In [8]:
def refined_risk_stratum(row):
    """Sensitivity-only stratum incorporating reported complications.

    Missing complication values are never treated as 0/no-complication —
    records with either value missing are explicitly excluded, not imputed.
    """
    if not row["first_birth"]:
        return "not_applicable_multiparous"
    if pd.isna(row["convulsions_during_pregnancy"]) or pd.isna(row["swelling_during_pregnancy"]):
        return "excluded_missing_complication_data"
    is_singleton = row["twin_order"] == 0
    is_young = row["maternal_age"] < 35
    has_complication = (
        (row["convulsions_during_pregnancy"] == 1) or (row["swelling_during_pregnancy"] == 1)
    )
    if is_young and is_singleton and not has_complication:
        return "refined_lower_risk"
    return "refined_elevated_risk"


df["risk_stratum_refined_sensitivity"] = df.apply(refined_risk_stratum, axis=1)

refined_counts = df["risk_stratum_refined_sensitivity"].value_counts()
refined_pct = df["risk_stratum_refined_sensitivity"].value_counts(normalize=True) * 100

print("Refined (sensitivity-only) stratum counts and percentages:")
for stratum in refined_counts.index:
    print(f"  {stratum}: {refined_counts[stratum]} ({refined_pct[stratum]:.2f}%)")

n_excluded = (df["risk_stratum_refined_sensitivity"] == "excluded_missing_complication_data").sum()
print(f"\nOf {n_first} first births, {n_excluded} ({n_excluded / n_first * 100:.2f}%) "
      f"excluded from the refined scheme due to missing complication data.")

Refined (sensitivity-only) stratum counts and percentages:
  not_applicable_multiparous: 118704 (58.97%)
  refined_lower_risk: 34881 (17.33%)
  refined_elevated_risk: 24452 (12.15%)
  excluded_missing_complication_data: 23274 (11.56%)

Of 82607 first births, 23274 (28.17%) excluded from the refined scheme due to missing complication data.


## 8. `s434`/`s435` missingness audit

Tests whether complication-data availability is systematic, across facility
sector, C-section outcome, ANC attendance, age, residence, wealth, education,
state, birth order (full cohort), and survey weight.

In [9]:
fb = df[df["first_birth"]].copy()
fb["complication_missing"] = (
    fb["convulsions_during_pregnancy"].isna() | fb["swelling_during_pregnancy"].isna()
)

print(f"First births with missing complication data: "
      f"{fb['complication_missing'].sum()} ({fb['complication_missing'].mean() * 100:.2f}%)")

First births with missing complication data: 23274 (28.17%)


In [10]:
def audit_categorical(column, label):
    """Prints missingness rate by category and a chi-square test of association."""
    print(f"--- {label} ({column}) ---")
    tab = pd.crosstab(fb[column], fb["complication_missing"], normalize="index") * 100
    tab.columns = ["pct_complete", "pct_missing"]
    print(tab.round(2))
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(fb[column], fb["complication_missing"]))
    print(f"Chi-square p-value: {p:.2e}\n")


def audit_numeric(column, label):
    """Compares means between complete and missing groups with a Mann-Whitney U test."""
    complete = fb.loc[~fb["complication_missing"], column].dropna()
    missing = fb.loc[fb["complication_missing"], column].dropna()
    print(f"--- {label} ({column}) ---")
    print(f"Mean (complete): {complete.mean():.3f} | Mean (missing): {missing.mean():.3f}")
    _, p = stats.mannwhitneyu(complete, missing, alternative="two-sided")
    print(f"Mann-Whitney U p-value: {p:.2e}\n")


audit_categorical("facility_type", "Facility sector")
audit_categorical("csection", "C-section outcome")
audit_numeric("anc_visits", "ANC visits (numeric)")

fb["any_anc"] = fb["anc_visits"].fillna(-1) > 0
print("--- ANC attendance (any visits vs none/missing) ---")
print((pd.crosstab(fb["any_anc"], fb["complication_missing"], normalize="index") * 100).round(2))
print()

audit_numeric("maternal_age", "Maternal age")
audit_categorical("residence", "Residence (urban/rural)")
audit_categorical("wealth_index", "Wealth index")
audit_numeric("education_years", "Education years")

print("--- State ---")
state_missing = fb.groupby("state")["complication_missing"].mean() * 100
print(f"Range across states: min={state_missing.min():.2f}%, "
      f"max={state_missing.max():.2f}%, std={state_missing.std():.2f}")
chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(fb["state"], fb["complication_missing"]))
print(f"Chi-square p-value: {p:.2e}\n")

print("--- Birth order (full cohort, not just first births) ---")
df["complication_missing_full"] = (
    df["convulsions_during_pregnancy"].isna() | df["swelling_during_pregnancy"].isna()
)
print((df.groupby("birth_order")["complication_missing_full"].mean() * 100).round(2))
print()

audit_numeric("sample_weight", "Survey weight (raw)")

--- Facility sector (facility_type) ---
               pct_complete  pct_missing
facility_type                           
other                 69.06        30.94
private               74.14        25.86
public                70.88        29.12
Chi-square p-value: 2.72e-20

--- C-section outcome (csection) ---
          pct_complete  pct_missing
csection                           
0                69.32        30.68
1                78.65        21.35
Chi-square p-value: 8.96e-154

--- ANC visits (numeric) (anc_visits) ---
Mean (complete): 5.518 | Mean (missing): 4.049
Mann-Whitney U p-value: 3.59e-02

--- ANC attendance (any visits vs none/missing) ---
complication_missing  False  True 
any_anc                           
False                 21.09  78.91
True                  99.94   0.06

--- Maternal age (maternal_age) ---
Mean (complete): 24.763 | Mean (missing): 25.239
Mann-Whitney U p-value: 6.36e-111

--- Residence (urban/rural) (residence) ---
           pct_complete  pct_miss

**Interpretation (fill in after running against your data — do not assume the
pattern is identical to a prior run):** if missingness varies significantly by
ANC attendance, socioeconomic/geographic variables, and — critically — by the
C-section outcome itself, this confirms complication-data availability is not
missing at random. Any refined-stratum result must then be reported as
describing the complete-case, ANC-attending subpopulation, not all first
births.

## 9. Final cohort, sector, and prevalence tables

In [11]:
print("Row count:", len(df))
print("Unique respondents:", df["respondent_id"].nunique())
print("\nRow count vs v1:", len(df), "vs", n_v1,
      "(UNCHANGED)" if len(df) == n_v1 else "(CHANGED -- investigate)")
print("Respondent count vs v1:", df["respondent_id"].nunique(), "vs", resp_v1,
      "(UNCHANGED)" if df["respondent_id"].nunique() == resp_v1 else "(CHANGED -- investigate)")

print("\nFacility sector counts:")
print(df["facility_type"].value_counts())

print(f"\nC-section prevalence (overall): {df['csection'].mean() * 100:.2f}%")

Row count: 201311
Unique respondents: 158429

Row count vs v1: 201311 vs 201311 (UNCHANGED)
Respondent count vs v1: 158429 vs 158429 (UNCHANGED)

Facility sector counts:
facility_type
public     150299
private     50495
other         517
Name: count, dtype: int64

C-section prevalence (overall): 22.23%


## 10. Feasibility table — `lower_risk` and `elevated_risk`

Reproduces the approved feasibility check: N, facility split, raw and
survey-weighted C-section prevalence by sector, for each primary stratum.

In [12]:
def stratum_feasibility_table(stratum_name):
    sub = df[df["risk_stratum_primary"] == stratum_name].copy()
    sub["w"] = sub["sample_weight_normalized"]

    print(f"===== {stratum_name.upper()} =====")
    print("Total N:", len(sub))
    print("Unique respondents:", sub["respondent_id"].nunique())

    fac_counts = sub["facility_type"].value_counts()
    fac_pct = sub["facility_type"].value_counts(normalize=True) * 100
    print("\nFacility counts / %:")
    for fac in fac_counts.index:
        print(f"  {fac}: {fac_counts[fac]} ({fac_pct[fac]:.2f}%)")

    print(f"\nC-section prevalence overall: {sub['csection'].mean() * 100:.2f}%")

    for fac in ["public", "private", "other"]:
        fac_sub = sub[sub["facility_type"] == fac]
        if len(fac_sub) == 0:
            continue
        prev = fac_sub["csection"].mean() * 100
        print(f"  {fac}: N={len(fac_sub)}, C-section prevalence={prev:.2f}%")

    print("\nSurvey-weighted prevalence by sector:")
    for fac in ["public", "private"]:
        fac_sub = sub[sub["facility_type"] == fac]
        if len(fac_sub) > 0 and fac_sub["w"].sum() > 0:
            weighted_prev = np.average(fac_sub["csection"], weights=fac_sub["w"]) * 100
            print(f"  {fac}: {weighted_prev:.2f}%")

    print("\nFacility x C-section cross-tab (sparse-cell check):")
    print(pd.crosstab(sub["facility_type"], sub["csection"]))
    print("=" * 60, "\n")


stratum_feasibility_table("lower_risk")
stratum_feasibility_table("elevated_risk")

===== LOWER_RISK =====
Total N: 79701
Unique respondents: 79701

Facility counts / %:
  public: 56781 (71.24%)
  private: 22750 (28.54%)
  other: 170 (0.21%)

C-section prevalence overall: 26.02%
  public: N=56781, C-section prevalence=16.71%
  private: N=22750, C-section prevalence=49.45%
  other: N=170, C-section prevalence=0.00%

Survey-weighted prevalence by sector:
  public: 17.57%
  private: 50.16%

Facility x C-section cross-tab (sparse-cell check):
csection           0      1
facility_type              
other            170      0
private        11501  11249
public         47295   9486

===== ELEVATED_RISK =====
Total N: 2906
Unique respondents: 2906

Facility counts / %:
  public: 1658 (57.05%)
  private: 1237 (42.57%)
  other: 11 (0.38%)

C-section prevalence overall: 50.79%
  public: N=1658, C-section prevalence=36.49%
  private: N=1237, C-section prevalence=70.41%
  other: N=11, C-section prevalence=0.00%

Survey-weighted prevalence by sector:
  public: 41.01%
  private: 70

## 11. Variable dictionary — exhaustive (one row per actual v2 column)

Every column that actually exists in `df_model_v2` gets exactly one row.
`v106`/`v131` are kept as two additional optional rows, explicitly marked as
not present, for a documented total of 55 rows (53 real + 2 optional).

In [13]:
# Domain/role/timing annotations reflect the approved v2 protocol, including
# the reclassification of m43/s420a-e into the ANC-service domain and the
# explicit exclusion of any prior-C-section derivation. Metadata below is
# reused from the verified v1 audit and notebook 01 outputs; anything not
# directly confirmed there is marked TODO rather than invented.
variable_dictionary_rows = [
    ("respondent_id", "caseid", "Respondent identifier", "string", "none observed", "survey_design", "grouping only", "grouping for cross-fitting", "design", "none"),
    ("birth_index", "bidx", "Birth history index", "integer", "none observed", "survey_design", "grouping only", "NOT used as parity proxy -- see documentation", "design", "none"),
    ("cluster_number", "v001", "Cluster number", "integer", "none observed; checked live vs primary_sampling_unit", "survey_design", "not a predictor", "cluster identifier", "design", "none"),
    ("household_number", "v002", "Household number", "integer", "none observed", "survey_design", "not a predictor", "n/a", "design", "none"),
    ("respondent_line_number", "v003", "Respondent line number", "integer", "none observed", "survey_design", "not a predictor", "n/a", "design", "none"),
    ("sample_weight", "v005", "Raw sample weight", "integer, unnormalized", "none observed", "survey_design", "not a predictor", "weighting (pre-normalization)", "design", "none"),
    ("primary_sampling_unit", "v021", "PSU", "integer", "none observed; checked live vs cluster_number", "survey_design", "not a predictor", "variance/cross-fitting groups", "design", "none"),
    ("sample_stratum_v022", "v022", "Candidate stratum A", "integer", "none observed; checked live vs v023", "survey_design", "not a predictor", "stratum for variance estimation", "design", "none"),
    ("sample_stratum_v023", "v023", "Candidate stratum B", "integer", "none observed; checked live vs v022", "survey_design", "not a predictor", "stratum for variance estimation", "design", "none"),
    ("state", "v024", "State", "categorical", "none observed", "survey_design", "not used (sensitivity candidate)", "geographic confounder candidate; driver of s434/s435 missingness", "design", "none"),
    ("district", "sdist", "District", "categorical", "none observed", "survey_design", "not used", "geographic sensitivity", "design", "none"),
    ("csection", "m17", "Delivery by Cesarean section", "0=no, 1=yes", "none observed", "outcome", "target", "outcome", "outcome", "n/a"),
    ("delivery_place_code", "m15", "Raw place-of-delivery code", "11 DHS categories", "none observed", "exposure_raw", "not used directly", "superseded by facility_type", "verify_timing", "low"),
    ("facility_type", "m15", "Facility sector (recoded)", "public/private/other", "none observed", "exposure", "exposure (B3/B4)", "exposure; 'other' excluded from private-vs-public contrast", "verify_timing", "low_as_exposure"),
    ("maternal_age", "v012", "Mother's current age", "continuous years", "none observed", "maternal_clinical", "predictor", "confounder", "pre_delivery", "none"),
    ("education_years", "v133", "Education, completed years", "continuous", "97 -> missing (checked live in this notebook)", "socioeconomic_geographic", "predictor", "confounder", "pre_delivery", "none"),
    ("wealth_index", "v190", "Household wealth index", "1-5 ordinal", "none observed", "socioeconomic_geographic", "predictor", "confounder", "pre_delivery", "none"),
    ("residence", "v025", "Urban/rural residence", "1=urban, 2=rural", "none observed", "socioeconomic_geographic", "predictor", "confounder", "pre_delivery", "none"),
    ("religion", "v130", "Religion", "categorical", "none observed", "socioeconomic_geographic", "predictor", "confounder", "pre_delivery", "none"),
    ("social_group", "s116", "Social group (India-specific)", "4 categories", "8 -> missing", "socioeconomic_geographic", "predictor", "confounder", "pre_delivery", "none"),
    ("health_insurance", "v481", "Health insurance coverage", "0/1", "none observed", "socioeconomic_geographic", "predictor", "confounder (coarse)", "pre_delivery", "none"),
    ("birth_order", "bord", "Birth order of this child", "integer >=1", "none observed", "maternal_clinical", "predictor", "confounder / stratifier basis", "pre_delivery", "none"),
    ("total_children_ever_born", "v201", "Total children ever born (survey date)", "integer", "none observed", "maternal_clinical", "predictor", "TODO: confirm timing vs analyzed birth before causal use", "TODO", "TODO"),
    ("preceding_birth_interval", "b11", "Preceding birth interval", "months", "TODO: confirm DHS special codes (e.g. 997/998)", "maternal_clinical", "predictor", "confounder", "pre_delivery", "none"),
    ("age_at_first_birth", "v212", "Age at first birth", "integer", "TODO: confirm special codes", "maternal_clinical", "predictor", "confounder", "pre_delivery", "none"),
    ("twin_order", "b0", "Plurality / twin order", "0=single, 1/2=multiple", "TODO: confirm special codes", "maternal_clinical", "predictor", "confounder / stratifier basis", "pre_delivery", "none"),
    ("bmi", "v445", "Maternal BMI", "continuous, /100 applied", "9998 -> missing", "maternal_clinical", "predictor", "confounder", "pre_delivery", "none"),
    ("first_anc_timing", "m13", "Month of first ANC visit", "integer", "TODO: confirm special codes", "anc_service", "predictor", "possible_mediator", "pre_delivery", "low"),
    ("anc_visits", "m14", "Number of ANC visits", "integer", "98 -> missing; undocumented 90-95 -> missing (v1)", "anc_service", "predictor", "possible_mediator; drives s434/s435 missingness", "pre_delivery", "low"),
    ("anc_doctor", "m2a", "ANC from doctor", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_nurse_midwife", "m2b", "ANC from nurse/midwife", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_traditional_attendant", "m2g", "ANC from traditional attendant", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_community_worker", "m2h", "ANC from community worker", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_anganwadi_worker", "m2i", "ANC from anganwadi worker", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_asha_worker", "m2j", "ANC from ASHA worker", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("anc_other_provider", "m2k", "ANC from other provider", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("no_anc_provider", "m2n", "No ANC provider", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("weight_measured_during_pregnancy", "m42a", "Weight measured during ANC", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("bp_measured_during_pregnancy", "m42c", "BP measured during ANC", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("urine_sample_during_pregnancy", "m42d", "Urine sample taken during ANC", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("blood_sample_during_pregnancy", "m42e", "Blood sample taken during ANC", "0/1", "none observed", "anc_service", "predictor", "mediator", "pre_delivery", "low"),
    ("told_about_pregnancy_complications", "m43", "Told about pregnancy complications (general)", "0/1", "8 -> missing", "anc_service [RECLASSIFIED from clinical_risk]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("counselled_vaginal_bleeding", "s420a", "Told about danger sign: vaginal bleeding", "0/1", "see v1 missingness", "anc_service [RECLASSIFIED]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("counselled_convulsions", "s420b", "Told about danger sign: convulsions", "0/1", "see v1 missingness", "anc_service [RECLASSIFIED]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("counselled_prolonged_labour", "s420c", "Told about danger sign: prolonged labour", "0/1", "see v1 missingness", "anc_service [RECLASSIFIED]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("counselled_severe_abdominal_pain", "s420d", "Told about danger sign: severe abdominal pain", "0/1", "see v1 missingness", "anc_service [RECLASSIFIED]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("counselled_high_blood_pressure", "s420e", "Told about danger sign: high blood pressure", "0/1", "see v1 missingness", "anc_service [RECLASSIFIED]", "predictor", "mediator, not confounder", "pre_delivery", "low"),
    ("convulsions_during_pregnancy", "s434", "Convulsions during pregnancy (occurrence)", "0/1", "8 -> missing; non-random missingness (see audit)", "maternal_clinical", "predictor", "confounder / sensitivity-stratifier basis ONLY", "pre_delivery", "non-random missingness"),
    ("swelling_during_pregnancy", "s435", "Swelling during pregnancy (occurrence)", "0/1", "8 -> missing; non-random missingness (see audit)", "maternal_clinical", "predictor", "confounder / sensitivity-stratifier basis ONLY", "pre_delivery", "non-random missingness"),
    ("sample_weight_normalized", "v005 / 1,000,000", "Normalized sample weight", "float", "n/a (derived)", "survey_design", "not a predictor", "weighting variable for B1/B4", "design", "none"),
    ("first_birth", "derived from birth_order", "First-birth flag (birth_order==1)", "boolean", "n/a (derived)", "maternal_clinical", "predictor / stratifier", "stratifier", "pre_delivery", "none"),
    ("risk_stratum_primary", "derived", "PRIMARY risk stratum (fully observed vars only)", "lower_risk / elevated_risk / not_applicable_multiparous", "n/a (derived); 0 unclassified", "maternal_clinical", "not used in B2", "primary stratifier for B4", "pre_delivery", "none"),
    ("risk_stratum_refined_sensitivity", "derived", "SENSITIVITY-ONLY stratum incl. complications", "refined_lower_risk / refined_elevated_risk / not_applicable_multiparous / excluded_missing_complication_data", "n/a (derived); see missingness audit", "maternal_clinical", "not used in B2", "sensitivity stratifier for B4 only -- NOT primary", "pre_delivery", "see missingness audit"),
    # Optional rows for variables NOT present in df_model_v2 -- kept for documentation only.
    ("education_level (v106)", "v106", "Education, categorical level", "NOT IN v2 -- requires raw NFHS-5 access", "n/a", "socioeconomic_geographic", "OPTIONAL robustness alternate to education_years", "OPTIONAL robustness alternate", "pre_delivery", "NOT YET AVAILABLE"),
    ("caste_tribe (v131)", "v131", "Caste/tribe (standard DHS)", "NOT IN v2 -- requires raw NFHS-5 access", "n/a", "socioeconomic_geographic", "OPTIONAL robustness alternate to social_group", "OPTIONAL robustness alternate", "pre_delivery", "NOT YET AVAILABLE"),
]

variable_dictionary = pd.DataFrame(
    variable_dictionary_rows,
    columns=[
        "column", "source_variable", "label", "coding", "missing_rules",
        "domain", "predictive_role", "causal_role", "timing", "leakage_status",
    ],
)

print("Variable dictionary rows:", len(variable_dictionary))
variable_dictionary.head()

Variable dictionary rows: 55


,column,source_variable,label,coding,missing_rules,domain,predictive_role,causal_role,timing,leakage_status
0,respondent_id,caseid,Respondent identifier,string,none observed,survey_design,grouping only,grouping for cross-fitting,design,none
1,birth_index,bidx,Birth history index,integer,none observed,survey_design,grouping only,NOT used as parity proxy -- see documentation,design,none
2,cluster_number,v001,Cluster number,integer,none observed; checked live vs primary_samplin...,survey_design,not a predictor,cluster identifier,design,none
3,household_number,v002,Household number,integer,none observed,survey_design,not a predictor,n/a,design,none
4,respondent_line_number,v003,Respondent line number,integer,none observed,survey_design,not a predictor,n/a,design,none


### Dictionary completeness assertions

Confirms — rather than assumes — that the dictionary is exhaustive: every
actual v2 column appears exactly once, there are no duplicate entries, and
any row describing a column NOT present in v2 is explicitly labeled as such.

In [14]:
df_columns = set(df.columns) - {"complication_missing_full"}  # audit-only helper, dropped before save
dict_columns_raw = variable_dictionary["column"].tolist()

# Rows for optional/unavailable variables use a "(vNNN)" suffix in this dictionary;
# strip that to get the actual candidate column name being described.
def base_column_name(name):
    return name.split(" (")[0]

dict_column_names = [base_column_name(c) for c in dict_columns_raw]

# 1. No duplicate dictionary entries.
duplicates = pd.Series(dict_columns_raw)[pd.Series(dict_columns_raw).duplicated()].tolist()
assert not duplicates, f"Duplicate dictionary entries found: {duplicates}"
print("No duplicate dictionary entries.")

# 2. Every actual v2 column appears exactly once in the dictionary.
dict_names_present_in_v2 = [
    base_column_name(row["column"]) for _, row in variable_dictionary.iterrows()
    if base_column_name(row["column"]) in df_columns
]
missing_from_dict = df_columns - set(dict_names_present_in_v2)
assert not missing_from_dict, f"v2 columns missing from dictionary: {missing_from_dict}"
assert len(dict_names_present_in_v2) == len(set(dict_names_present_in_v2)), \
    "A v2 column is documented more than once in the dictionary."
assert len(dict_names_present_in_v2) == len(df_columns), \
    f"Dictionary covers {len(dict_names_present_in_v2)} actual columns, expected {len(df_columns)}."
print(f"Every one of the {len(df_columns)} actual v2 columns appears exactly once in the dictionary.")

# 3. Any dictionary row describing a column NOT in v2 must say so explicitly.
optional_marker = "NOT IN v2"
for _, row in variable_dictionary.iterrows():
    col_name = base_column_name(row["column"])
    if col_name not in df_columns:
        assert optional_marker in row["coding"] or "unavailable" in row["causal_role"].lower() \
            or "OPTIONAL" in row["predictive_role"], (
            f"Row for '{row['column']}' is not in v2 but is not explicitly marked "
            f"optional/unavailable."
        )
print("All dictionary rows for columns absent from v2 are explicitly marked optional/unavailable.")

n_actual = sum(1 for c in dict_columns_raw if base_column_name(c) in df_columns)
n_optional = len(dict_columns_raw) - n_actual
print(f"\nDictionary rows for actual v2 columns: {n_actual} (expected 53)")
print(f"Optional/unavailable rows: {n_optional} (expected 2)")
print(f"Total dictionary rows: {len(variable_dictionary)} (expected 55)")

No duplicate dictionary entries.
Every one of the 53 actual v2 columns appears exactly once in the dictionary.
All dictionary rows for columns absent from v2 are explicitly marked optional/unavailable.

Dictionary rows for actual v2 columns: 53 (expected 53)
Optional/unavailable rows: 2 (expected 2)
Total dictionary rows: 55 (expected 55)


## 12. Save `df_model_v2` and the variable dictionary

In [15]:
# complication_missing_full was an audit-only helper column (Section 8) used to
# check missingness by birth order across the full cohort. It is not part of the
# frozen v2 schema and must not be saved.
df_v2 = df.drop(columns=["complication_missing_full"], errors="ignore")

assert len(df_v2) == n_v1, "Row count changed unexpectedly -- stop and investigate before saving."
assert df_v2["respondent_id"].nunique() == resp_v1, "Respondent count changed unexpectedly -- stop and investigate."

V2_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_v2.to_parquet(V2_OUT_PATH, engine="pyarrow", index=False)
variable_dictionary.to_csv(DICT_OUT_PATH, index=False)

print("Saved:", V2_OUT_PATH.resolve())
print("Shape:", df_v2.shape)
print("\nSaved:", DICT_OUT_PATH.resolve())
print("Shape:", variable_dictionary.shape)

Saved: C:\Users\Tushna\Downloads\IPD\IPD\IPD\data\processed\df_model_v2.parquet
Shape: (201311, 53)

Saved: C:\Users\Tushna\Downloads\IPD\IPD\IPD\data\processed\variable_dictionary_v2.csv
Shape: (55, 10)


## 13. Documentation — key v2 design decisions and why

**Why prior C-section was dropped.**
NFHS-5's delivery module (`m17`, which records Cesarean delivery, and `m15`,
place of delivery) is only administered for births within a limited recall
window — not for a woman's full birth history. Coverage of `m17` collapses
sharply with birth order (highest at `bidx`=1, falling to near-zero by
`bidx`=4), so for any analyzed birth with `birth_order >= 2`, whether the
*preceding* birth was a C-section is missing in the large majority of cases.
That missingness means "not asked," not "vaginal delivery" — treating it as
"no prior C-section" would silently convert unknown history into a false
negative, concentrated exactly among the higher-parity women any such
stratifier would be trying to isolate. `v401` ("last birth a caesarean
section") exists in the metadata but was never evaluated in v1 and likely has
the same one-birth-only coverage limitation; it remains a documentation-only
follow-up item, not a variable used anywhere in this notebook.

**Why `bord` (birth_order) is used instead of `bidx` (birth_index) for
identifying first births.**
`bidx` is the row-position of a birth within the recorded birth-history array
— a bookkeeping index that can diverge from true birth order when non-live
outcomes are interspersed in a woman's history. `v201` (total children ever
born) is a survey-date snapshot and can overstate parity at the time of an
earlier analyzed birth if the woman had more children afterward. `bord` is
DHS's own computed birth-order-of-this-child variable, so `bord == 1` is a
direct, verifiable fact about that specific birth, not an inference from
incomplete history — exactly what a first-birth flag requires.

**Why complication variables (`s434`/`s435`) are sensitivity-only, not part of
the primary stratification.**
The missingness audit above shows complication-data availability is not
missing at random: it is driven almost mechanically by ANC attendance,
carries a real socioeconomic/geographic gradient (wealth, education, state,
urban/rural), and — most importantly — is correlated with the outcome itself
(C-section cases have less missing complication data than vaginal births).
Restricting the primary analysis to complication-observed records would mean
analyzing a non-random, outcome-entangled subpopulation. The primary strata
therefore use only variables with zero missingness (`maternal_age`,
`twin_order`, `birth_order`); the complication-augmented definition is
retained strictly as a labeled sensitivity check.

**Why `v106`/`v131` are deferred.**
`education_level` (`v106`) and `caste_tribe` (`v131`) are not present in
`df_model_v1.parquet` and require a fresh pull from the raw NFHS-5 `.DTA`
file, which needs DHS data access. They were only ever proposed as
DAG-robustness alternates to `education_years` and `social_group`, both of
which are already present in v1/v2 with low-to-zero missingness and serve as
complete primary confounders for those two domains. Their absence therefore
does not block any primary-analysis work in this notebook; they remain
documented in the variable dictionary as optional future additions.